In [1]:
import pandas as pd

df = pd.read_csv("SMSSpamCollection", sep="\t", names=["label", "message"])

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Encode Labels

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

df['label'].value_counts()

,count
label,
0,4825
1,747


TF-IDF Vectorization

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_text = df['message']
y = df['label']

tfidf = TfidfVectorizer(stop_words='english')

X = tfidf.fit_transform(X_text)

print("Shape of TF-IDF matrix:", X.shape)

Shape of TF-IDF matrix: (5572, 8444)


Define Base Models

In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

nb = MultinomialNB()

lr = LogisticRegression(max_iter=1000)

# LinearSVC does NOT give probabilities so we calibrate it
svm = CalibratedClassifierCV(LinearSVC())

Function to Evaluate Any Model (K-Fold)

In [6]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model, X, y, model_name):

    precision_list = []
    recall_list = []
    f1_list = []
    auc_list = []

    for train_idx, test_idx in skf.split(X, y):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        precision_list.append(precision_score(y_test, preds))
        recall_list.append(recall_score(y_test, preds))
        f1_list.append(f1_score(y_test, preds))

        # Some models may not support predict_proba
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(X_test)[:, 1]
            auc_list.append(roc_auc_score(y_test, probs))
        else:
            auc_list.append(roc_auc_score(y_test, preds))

    print(f"\n{model_name} Results:")
    print("Precision:", np.mean(precision_list))
    print("Recall   :", np.mean(recall_list))
    print("F1 Score :", np.mean(f1_list))
    print("ROC-AUC  :", np.mean(auc_list))

Evaluate models

In [7]:
evaluate_model(nb, X, y, "Naive Bayes")
evaluate_model(lr, X, y, "Logistic Regression")
evaluate_model(svm, X, y, "Linear SVM")


Naive Bayes Results:
Precision: 1.0
Recall   : 0.7804116331096197
F1 Score : 0.8765615424853863
ROC-AUC  : 0.9894953622886021

Logistic Regression Results:
Precision: 0.9899723370395292
Recall   : 0.6612796420581656
F1 Score : 0.7924435088231784
ROC-AUC  : 0.9904911546174265

Linear SVM Results:
Precision: 0.9714881364379
Recall   : 0.9089306487695751
F1 Score : 0.93889042795506
ROC-AUC  : 0.9924822663467445


Import & Define Voting Classifiers

In [8]:
from sklearn.ensemble import VotingClassifier

# Hard Voting
hard_voting = VotingClassifier(
    estimators=[('nb', nb), ('lr', lr), ('svm', svm)],
    voting='hard'
)

# Soft Voting
soft_voting = VotingClassifier(
    estimators=[('nb', nb), ('lr', lr), ('svm', svm)],
    voting='soft'
)

Evaluate Voting Models

In [9]:
evaluate_model(hard_voting, X, y, "Hard Voting")
evaluate_model(soft_voting, X, y, "Soft Voting")


Hard Voting Results:
Precision: 0.9917424869037774
Recall   : 0.8058344519015661
F1 Score : 0.889006984819509
ROC-AUC  : 0.9023990912357572

Soft Voting Results:
Precision: 0.9891779043591317
Recall   : 0.8527158836689038
F1 Score : 0.9157697170732757
ROC-AUC  : 0.9930732691170846


Stacking + AdaBoost

In [10]:
from sklearn.ensemble import StackingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Stacking (meta-learner = Logistic Regression)
stacking = StackingClassifier(
    estimators=[('nb', nb), ('lr', lr), ('svm', svm)],
    final_estimator=LogisticRegression()
)

# AdaBoost with decision stumps
stump = DecisionTreeClassifier(max_depth=1)

adaboost = AdaBoostClassifier(
    estimator=stump,
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)

Evaluate models

In [11]:
evaluate_model(stacking, X, y, "Stacking")
evaluate_model(adaboost, X, y, "AdaBoost (Stumps)")


Stacking Results:
Precision: 0.976144592931182
Recall   : 0.9196599552572706
F1 Score : 0.9467196168656498
ROC-AUC  : 0.9930402939574133

AdaBoost (Stumps) Results:
Precision: 0.954423352006823
Recall   : 0.42173601789709175
F1 Score : 0.5847982085244927
ROC-AUC  : 0.9263419967312307


Train Final Stacking Model

In [12]:
stacking.fit(X, y)

final_preds = stacking.predict(X)
final_probs = stacking.predict_proba(X)[:, 1]

Confusion Matrix

In [13]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, final_preds)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[4820    5]
 [   7  740]]


Create Comparison CSV

In [14]:
import pandas as pd

comparison_data = {
    "Model": [
        "NaiveBayes",
        "LogisticRegression",
        "LinearSVM",
        "HardVoting",
        "SoftVoting",
        "Stacking",
        "AdaBoost_Stumps"
    ],
    "F1_Score": [
        0.8765615424853863,
        0.7924435088231784,
        0.93889042795506,
        0.889006984819509,
        0.9157697170732757,
        0.9467196168656498,
        0.5847982085244927
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv("ensemble_comparison.csv", index=False)

comparison_df

,Model,F1_Score
0,NaiveBayes,0.876562
1,LogisticRegression,0.792444
2,LinearSVM,0.938890
3,HardVoting,0.889007
4,SoftVoting,0.915770
5,Stacking,0.946720
6,AdaBoost_Stumps,0.584798


Create Final Predictions CSV

In [15]:
final_output = pd.DataFrame({
    "MessageId": range(len(df)),
    "Actual": y,
    "Predicted": final_preds,
    "Probability": final_probs
})

final_output.to_csv("final_model_predictions.csv", index=False)

final_output.head()

,MessageId,Actual,Predicted,Probability
0,0,0,0,0.008042
1,1,0,0,0.007635
2,2,1,1,0.999831
3,3,0,0,0.007661
4,4,0,0,0.007614


Download output

In [16]:
from google.colab import files
files.download("ensemble_comparison.csv")
files.download("final_model_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Best Model → Stacking
- Highest F1
- Highest overall stability
- Excellent confusion matrix
- Very balanced precision & recall

AdaBoost (stumps) performed poorly because:
- Decision stumps are too weak for high-dimensional TF-IDF features.
- Boosting with depth-1 trees struggles in sparse text space.

Recommendation:

Among all models, Stacking achieved the best overall performance with the highest F1-score and strong ROC-AUC, showing better balance between precision and recall compared to individual classifiers and voting methods. While Linear SVM performed well as a standalone model, stacking effectively combined multiple classifiers to improve generalization. AdaBoost with decision stumps performed poorly due to the weakness of shallow trees in handling high-dimensional TF-IDF features. Therefore, stacking is recommended as the most effective combining strategy for this spam classification task.